# 03-2. 確率分布 — 動かして確かめる

📖 解説: [`../02_distributions.md`](../02_distributions.md)

## このノートで触るもの
1. 正規分布 $\mathcal{N}(\mu, \sigma^2)$ のサンプリングと PDF/CDF
2. 二項分布 $\mathrm{Bin}(n, p)$
3. ポアソン分布 $\mathrm{Poisson}(\lambda)$
4. 指数分布 $\mathrm{Exp}(\lambda)$
5. 中心極限定理を可視化

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (02_distributions.md)](../02_distributions.md)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", message=".*distutils Version classes.*", category=DeprecationWarning)
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)
from scipy import stats
from ipywidgets import interact, FloatSlider, IntSlider

%matplotlib inline
rng = np.random.default_rng(seed=42)

## 1. 正規分布

$$
p(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)
$$

対応するコード: `stats.norm.pdf(x, loc=μ, scale=σ)`

In [ ]:
# サンプリング
samples = rng.normal(loc=0, scale=1, size=10_000)
print(f'平均 μ̂      = {samples.mean():.4f}  (理論 0)')
print(f'標準偏差 σ̂  = {samples.std():.4f}  (理論 1)')
print(f'PDF p(0)    = {stats.norm.pdf(0):.4f}  (理論 1/√(2π) = {1/np.sqrt(2*np.pi):.4f})')
print(f'P(X ≤ 1.96) = {stats.norm.cdf(1.96):.4f}  (≈ 0.975)')

In [ ]:
def plot_normal(mu: float, sigma: float) -> None:
    """正規分布 N(μ, σ²) の PDF とサンプル ヒストグラム."""
    x = np.linspace(-10, 10, 200)
    pdf = stats.norm.pdf(x, loc=mu, scale=sigma)
    samples = rng.normal(mu, sigma, size=5000)
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(samples, bins=50, density=True, alpha=0.5, label='サンプル')
    ax.plot(x, pdf, 'r-', linewidth=2, label=f'PDF: N({mu}, {sigma}²)')
    ax.axvline(mu, color='red', linestyle='--', alpha=0.5)
    ax.set_title(f'$\\mu = {mu}$,  $\\sigma = {sigma}$')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()

interact(plot_normal,
         mu=FloatSlider(min=-3, max=3, step=0.5, value=0),
         sigma=FloatSlider(min=0.5, max=3, step=0.1, value=1));

## 2. 二項分布

$$
P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}
$$

$\mathbb{E}[X] = np$, $\mathrm{Var}[X] = np(1-p)$

対応するコード: `stats.binom.pmf(k, n, p)`

In [ ]:
n, p = 10, 0.5
ks = np.arange(n + 1)
pmf = stats.binom.pmf(ks, n, p)

for k, prob in zip(ks, pmf):
    bar = '█' * int(prob * 100)
    print(f'P(X = {k:2d}) = {prob:.4f}  {bar}')

print(f'\n平均 (理論)  = np = {n*p}')
print(f'分散 (理論)  = np(1-p) = {n*p*(1-p)}')

## 3. ポアソン分布

$$
P(X = k) = \frac{\lambda^k e^{-\lambda}}{k!}
$$

$\mathbb{E}[X] = \mathrm{Var}[X] = \lambda$

対応するコード: `stats.poisson.pmf(k, λ)`

In [ ]:
def plot_poisson(lam: float) -> None:
    """ポアソン分布の PMF を可視化."""
    ks = np.arange(0, max(20, int(lam * 3)))
    pmf = stats.poisson.pmf(ks, lam)
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(ks, pmf, alpha=0.7, edgecolor='black')
    ax.set_title(f'$\\mathrm{{Poisson}}(\\lambda = {lam})$:  $\\mathbb{{E}}[X] = \\mathrm{{Var}}[X] = {lam}$')
    ax.set_xlabel('k')
    ax.set_ylabel('P(X = k)')
    ax.grid(True, alpha=0.3)
    plt.show()

interact(plot_poisson, lam=FloatSlider(min=0.5, max=15, step=0.5, value=3));

## 4. 指数分布

$$
p(t) = \lambda e^{-\lambda t}
$$

ポアソン過程で「次のイベントまでの時間」。$\mathbb{E}[T] = 1/\lambda$

In [ ]:
lam = 3.0
samples = rng.exponential(scale=1/lam, size=10_000)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(samples, bins=50, density=True, alpha=0.5, label='サンプル')
t = np.linspace(0, 3, 100)
ax.plot(t, lam * np.exp(-lam * t), 'r-', label=f'$p(t) = \\lambda e^{{-\\lambda t}}$, $\\lambda={lam}$')
ax.set_xlabel('t (待ち時間)')
ax.set_title(f'$\\mathrm{{Exp}}({lam})$:  平均待ち時間 $= 1/\\lambda = {1/lam:.3f}$')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print(f'平均 (実測): {samples.mean():.4f}  (理論 1/λ = {1/lam:.4f})')

## 5. 中心極限定理 — 何度サンプリングしても結果は正規分布

$$
\sqrt{n}\left(\frac{1}{n}\sum_i X_i - \mu\right) \xrightarrow{d} \mathcal{N}(0, \sigma^2)
$$

**どんな分布から取ったサンプルでも、十分な数を平均すれば正規分布になる**

In [ ]:
def clt_demo(n_per_set: int, distribution: str) -> None:
    """様々な分布から n 個サンプル → 平均、を 5000 セット繰り返す."""
    n_sets = 5000
    
    if distribution == 'uniform':
        sample_means = np.array([rng.uniform(0, 1, n_per_set).mean() for _ in range(n_sets)])
        title_dist = 'Uniform(0, 1)'
    elif distribution == 'exponential':
        sample_means = np.array([rng.exponential(1.0, n_per_set).mean() for _ in range(n_sets)])
        title_dist = 'Exp(1)'
    elif distribution == 'binomial':
        sample_means = np.array([rng.binomial(1, 0.3, n_per_set).mean() for _ in range(n_sets)])
        title_dist = 'Bernoulli(0.3)'
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(sample_means, bins=50, density=True, alpha=0.6, color='skyblue')
    # 正規分布フィット
    mu, sigma = sample_means.mean(), sample_means.std()
    x_range = np.linspace(mu - 4*sigma, mu + 4*sigma, 100)
    ax.plot(x_range, stats.norm.pdf(x_range, mu, sigma), 'r-', linewidth=2, label='正規分布フィット')
    ax.set_title(f'{title_dist} から {n_per_set}個ずつサンプル → 平均 (5000セット)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()

interact(clt_demo,
         n_per_set=IntSlider(min=2, max=200, step=2, value=30),
         distribution=['uniform', 'exponential', 'binomial']);

→ 元の分布が**何でも**、$n$ を増やすと平均は釣り鐘型 (正規分布) になる!
これが **中心極限定理** の威力。

## まとめ
- 正規分布: 中心極限定理により至るところに現れる
- 二項分布: $n$ 回試行の成功数
- ポアソン分布: 単位時間あたりの稀な事象
- 指数分布: 待ち時間
- 中心極限定理: 平均は (元の分布によらず) 正規分布になる

## 次へ
→ [`03_expectation_variance.ipynb`](03_expectation_variance.ipynb)  解説 → [`../03_expectation_variance.md`](../03_expectation_variance.md)

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [`01_probability_basics.ipynb`](01_probability_basics.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`03_expectation_variance.ipynb`](03_expectation_variance.ipynb) |